# Multiproduct formula validation

この notebook は、MPF 実装の回路規約を小規模な厳密計算で確認します。

- $M(\Delta)=\sum_j a_j S_2(\Delta/k_j)^{k_j}$ の係数と identity padding
- 増幅前の zero-branch block が $M(\Delta)/2$ であること
- 1回の3-step robust OAA後に成功確率がほぼ1になること
- ideal MPFに対するLow–Kliuchnikov–Wiebe boundと、共有ancilla上で反復する増幅stepのscopeの違い
- 増幅stepのsegment数依存性と2量子ビットTFIMでの厳密比較
- 負の時刻と $t=0$

密行列はこの検証 notebook だけで使い、回路 builder は密行列を構築しません。

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from qiskit.quantum_info import Operator

from hamiltonian_resources import (
    PauliHamiltonian,
    build_multiproduct_circuit,
    build_trotter_circuit,
    compare_with_exact,
    multiproduct_coefficients,
    optimal_mpf_exponents,
    transverse_field_ising,
)

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=6, suppress=True)

In [ ]:
def zero_ancilla_block(circuit, system_qubits):
    """Extract <0...0|U|0...0> using Qiskit's little-endian ordering."""
    unitary = Operator(circuit).data
    ancillas = circuit.num_qubits - system_qubits
    selected = np.arange(2**system_qubits) * 2**ancillas
    return unitary[np.ix_(selected, selected)]


def classical_mpf_step(hamiltonian, step_time, m, schedule="new"):
    return sum(
        coefficient
        * Operator(
            build_trotter_circuit(hamiltonian, step_time, exponent, order=2)
        ).data
        for coefficient, exponent in zip(
            multiproduct_coefficients(m, schedule=schedule),
            optimal_mpf_exponents(m, schedule=schedule),
            strict=True,
        )
    )


def success_range(block):
    probabilities = np.linalg.eigvalsh(block.conj().T @ block)
    return float(probabilities.min()), float(probabilities.max())

## 1. 係数1-normと相殺identity padding

両scheduleは $\lambda=\sum_j|a_j|<2$ を満たします。`new`はOAA込みquery数を抑え、`legacy`は係数1-normを小さく保ちます。残りの $2-\lambda$ は、同じ大きさで符号が逆の2個のidentity branchへ分けるため、LCUの値を変えず正規化だけを2にできます。

In [ ]:
normalization_rows = []
for schedule in ("new", "legacy"):
    for m in (2, 3, 4, 5):
        coefficients = multiproduct_coefficients(m, schedule=schedule)
        coefficient_l1 = float(np.sum(np.abs(coefficients)))
        exponents = optimal_mpf_exponents(m, schedule=schedule)
        normalization_rows.append(
            {
                "schedule": schedule,
                "m": m,
                "exponents": exponents,
                "sum(k)": sum(exponents),
                "coefficient 1-norm": coefficient_l1,
                "padding weight": 2 - coefficient_l1,
                "LCU normalization": 2.0,
            }
        )
display(pd.DataFrame(normalization_rows))

## 2. 単一stepのzero blockとrobust OAA

非可換な1量子ビットHamiltonianで、増幅前blockを $B=M/2$ と比較し、増幅後blockを厳密な三次式 $3B-4BB^\dagger B$ と比較します。MPFそのものとの差は、MPF stepのunitarity defectに由来します。

In [ ]:
hamiltonian_1q = PauliHamiltonian.from_terms(
    1, [("X", 0.6), ("Z", -0.4)]
)
step_time = 0.7
m = 3
target_step = classical_mpf_step(hamiltonian_1q, step_time, m)
target_before = target_step / 2
target_after = 3 * target_before - 4 * target_before @ target_before.conj().T @ target_before
unamplified = build_multiproduct_circuit(
    hamiltonian_1q, step_time, m=m, amplitude_amplification=False
)
amplified = build_multiproduct_circuit(hamiltonian_1q, step_time, m=m)
block_before = zero_ancilla_block(unamplified, 1)
block_after = zero_ancilla_block(amplified, 1)
before_success = success_range(block_before)
after_success = success_range(block_after)

display(
    pd.DataFrame(
        [
            {
                "circuit": "before OAA",
                "block identity error": np.linalg.norm(block_before - target_before, 2),
                "distance from M": np.linalg.norm(block_before - target_step, 2),
                "minimum success": before_success[0],
                "maximum success": before_success[1],
            },
            {
                "circuit": "after OAA",
                "block identity error": np.linalg.norm(block_after - target_after, 2),
                "distance from M": np.linalg.norm(block_after - target_step, 2),
                "minimum success": after_success[0],
                "maximum success": after_success[1],
            },
        ]
    )
)

## 3. segment数に対する時間発展誤差

各点で $\Delta=t/r$ のMPF stepをOAAして、同じbranch register上で $r$ 回適用します。ancilla漏れを含むunitaryの反復なので、最終zero blockを単純な $M(\Delta)^r$ とは同一視せず、厳密時間発展へのstate errorと最終成功確率を確認します。

In [ ]:
rng = np.random.default_rng(2026)
initial_state = rng.normal(size=4) + 1j * rng.normal(size=4)
initial_state /= np.linalg.norm(initial_state)
hamiltonian_2q = transverse_field_ising(2, coupling=1.0, field=0.7)
total_time = 0.6
segment_rows = []
for segments in (1, 2, 3, 4, 6):
    result = compare_with_exact(
        hamiltonian_2q,
        total_time,
        method="multiproduct",
        initial_state=initial_state,
        reps=segments,
        mpf_m=2,
    )
    segment_rows.append({"segments": segments, **result})
segment_frame = pd.DataFrame(segment_rows)
display(segment_frame[["segments", "state_error", "fidelity", "success_probability"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].loglog(segment_frame["segments"], segment_frame["state_error"], "o-")
axes[0].set(xlabel="segments r", ylabel="phase-aligned state error")
axes[1].plot(segment_frame["segments"], segment_frame["success_probability"], "o-")
axes[1].set(xlabel="segments r", ylabel="all-zero branch probability", ylim=(0.99, 1.0001))
fig.tight_layout()

## 4. 2量子ビットTFIMで次数とsegment数を比較

同じランダム初期状態を使い、$m$ と $r$ を変えたときの厳密時間発展との差を表示します。

In [ ]:
tfim_rows = []
for m in (2, 3):
    for segments in (1, 2, 4):
        result = compare_with_exact(
            hamiltonian_2q,
            0.3,
            method="multiproduct",
            initial_state=initial_state,
            reps=segments,
            mpf_m=m,
        )
        tfim_rows.append({"m": m, "segments": segments, **result})
display(
    pd.DataFrame(tfim_rows)[
        ["m", "segments", "state_error", "fidelity", "success_probability"]
    ]
)

## 5. 負の時刻とゼロ時刻

負の時刻では正の時刻のadjointが得られ、ゼロ時刻はancillaを持たないsystem-only恒等回路になります。

In [ ]:
forward = build_multiproduct_circuit(hamiltonian_1q, 0.1, m=2, segments=2)
backward = build_multiproduct_circuit(hamiltonian_1q, -0.1, m=2, segments=2)
zero_time = build_multiproduct_circuit(hamiltonian_1q, 0.0, m=2, segments=2)
edge_summary = {
    "negative-time adjoint error": np.linalg.norm(
        zero_ancilla_block(backward, 1)
        - zero_ancilla_block(forward, 1).conj().T,
        2,
    ),
    "zero-time qubits": zero_time.num_qubits,
    "zero-time identity error": np.linalg.norm(Operator(zero_time).data - np.eye(2), 2),
    "zero-time construction": zero_time.metadata["construction"],
}
edge_summary

## 結論

この実装では、new/legacy schedule、正規化2、OAAの厳密な三次block変換、共有ancilla上のsegment反復を別々に検査できます。ここで確認している回路構成の正しさは、誤差定理のscopeとは区別されます。

既定のsegment数はLow–Kliuchnikov–Wiebe Eq. (16)の`low-rigorous`規則で選ばれ、Eqs. (14)–(15)はideal MPF operator $M(t/r)^r$を厳密にboundします。実装されたrobust-OAA共有ancilla回路については別の合成定理をまだ実装していないため、`bound_scope=ideal-mpf`かつ`circuit_bound_rigorous=False`です。以前のW2校正規則は`legacy-w2-proxy`として再現できますが、常に非厳密です。schedule の精度差はセクション 4 と `test_new_and_legacy_schedules_have_comparable_accuracy` で検査され、`new` は `legacy` と同一 order の誤差で `sum(k)` を削減します。